In [ ]:
! pip install peft transformers datasets

In [ ]:
## importing dataset
from transformers import AutoModelForCausalLM , AutoTokenizer , Pipeline
from peft import PeftModel , LoraConfig

In [ ]:
## creata easy sample
def create_sample(example):
  return {"role":"user","content":example}

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'[INFO] Device : {device}')

In [ ]:
# Our fine-tuned model will assign tags to text so we can easily filter them by type in the future
tags_dict = {'np': 'nutrition_panel',
 'il': 'ingredient_list',
 'me': 'menu',
 're': 'recipe',
 'fi': 'food_items',
 'di': 'drink_items',
 'fa': 'food_advertistment',
 'fp': 'food_packaging'}

In [ ]:
from google.colab import userdata
import os
from huggingface_hub import login

# Fetch the token safely from Colab Secrets and authenticate
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
!pip install --upgrade torchao # to solve version error

In [ ]:
BASE_MODEL = "google/gemma-3-270m-it"
MODEL_NAME = "RahulKate-173/FoodExtract-gemma-3-270m-fine-tune-peft-v2"
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path = BASE_MODEL ,
    dtype="auto",
    device_map = "auto",
    attn_implementation = "eager"

)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path = MODEL_NAME )
peft_model = PeftModel.from_pretrained(base_model,MODEL_NAME) # here the base mdoel is converted to peft model
## model and tokenizer loading complete

In [ ]:
## importing dataset for infernce
from datasets import load_dataset
dataset = load_dataset("mrdbourke/FoodExtract-1k")
dataset

In [ ]:
dataset["train"][42]

In [ ]:
import random
def create_random_index():
  rand_index = random.randint(0,len(dataset["train"])-1)
  return rand_index

In [ ]:
## creating pipeline
from transformers import pipeline
loaded_model_pipeline = pipeline("text-generation",model=peft_model,tokenizer=tokenizer)
random_sample = dataset["train"][create_random_index()] # random_sample created
prompt = create_sample(random_sample["sequence"]) # prompt format
updated_prompt = loaded_model_pipeline.tokenizer.apply_chat_template(conversation = [prompt] ,tokenize=False,add_generation_prompt=True)
default_outputs = loaded_model_pipeline(text_inputs = updated_prompt,max_new_tokens = 256)
print(f"[INFO] Input :\n {random_sample["sequence"]}")
print("[INFO] Outputs generated!!")
print(f"[INFO] Output :\n{default_outputs[0]["generated_text"][len(updated_prompt):]}")